In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
load_dotenv()

model = ChatOpenAI()

True

In [9]:
class BlogPost(TypedDict):
    title: str
    outline: str
    content: str


def create_outline(state: BlogPost) -> BlogPost:
    # fetch title
    title = state['title']
    # call llm
    prompt = f"Create an outline for a blog post with the title: {title}"
    content = model.invoke(prompt)

    # update in state
    state['outline'] = content
    return state

def create_blog(state: BlogPost) -> BlogPost:
    title = state['title']
    outline = state['outline']

    prompt = f"Create a blog post with the title: {title} and the following outline: {outline}"
    content = model.invoke(prompt)

    state['content'] = content
    return state


In [ ]:
graph = StateGraph(BlogPost)


graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [ ]:
initial_state = {"title": "The Future of AI in Healthcare"}
final_state = workflow.invoke(initial_state)